# Generate a photorealistic synthetic obstruction dataset with Gemini

Runs in **Google Colab**, no local `gcloud` install needed (Colab handles Vertex AI auth for you).

Per il TODO (sez. P1, "VERIFICA (piano di Andrea)"): calcolare il mAP anche su un dataset sintetico costruito con Gemini, in aggiunta al copy-paste e al test set reale `poli ingegneria`. A differenza del copy-paste (incolla un ritaglio scontornato -> bordi/luce non sempre coerenti), qui si parte da uno sfondo REALE (`Dataset/non_ostruite/porte|corridoi`, gli stessi usati da `pipeline3/src/generate_synthetic_dataset.py`) e si chiede al modello di image-editing di Gemini (`gemini-2.5-flash-image`, nickname "nano banana") di inserire da 1 a `MAX_OBJECTS` ostacoli (carrello, sedia a rotelle, barella, scatole, cestino) che blocchino il passaggio, mantenendo il resto della scena invariato.

Gemini **non** restituisce bounding box affidabili: questo notebook produce solo immagini + metadata (categoria richiesta, sfondo sorgente, prompt). Le box vanno disegnate a mano dopo, in locale, con lo stesso tool usato per il test set poli.

**Vincoli per il riuso come test set P2/P3** (oltre a P1):
- l'immagine editata deve avere ESATTAMENTE le stesse dimensioni pixel dell'immagine di input e la scena originale non deve essere ricomposta/ruotata/riscalata: solo l'ostacolo va aggiunto. Necessario perche' P3 confronta patch per patch contro la memory bank costruita sull'immagine normale corrispondente - se il framing cambia, l'anomaly map si sballa anche dove non c'e' nessun ostacolo.
- **una sola immagine editata per sfondo** (mai lo stesso sfondo due volte - puo' contenere da 1 a `MAX_OBJECTS` ostacoli insieme), e il file va salvato con lo STESSO NOME dell'immagine sorgente (`porta_093.jpg` -> `images/porte/porta_093.jpg`, non `porta_093_carrello_gemini.jpg`). Serve per sapere sempre, senza ambiguita', quale immagine normale/reference corrisponde a quale test occluso: quel nome e' la chiave per costruire la memory bank di P3 (reference = l'originale in `Dataset/non_ostruite/<venue>/`) e per collegare correttamente `reference_frame_id` quando si carica nel DB.
- le immagini sono organizzate per VENUE (`images/porte/`, `images/corridoi/`) cosi' da poter applicare il gating ROI di P3 (specifico per le porte) senza dover filtrare a mano.

## 1. Install dependencies

In [ ]:
!pip install -q google-genai

## 2. Authenticate

This opens a browser login and sets up Application Default Credentials inside the Colab VM (equivalent to `gcloud auth application-default login`, but with nothing to install).

In [ ]:
from google.colab import auth
auth.authenticate_user()

VERTEX_PROJECT = "your-gcp-project-id"  # <-- EDIT ME: your GCP project id
VERTEX_LOCATION = "us-central1"

## 3. Upload the background images (one venue at a time)

On your PC, zip `Dataset/non_ostruite/porte` and `Dataset/non_ostruite/corridoi` SEPARATELY (two zips). Each zip can contain the images directly or inside a subfolder - either way only the image files are kept, flattened into the right venue folder.

Run the helper cell once, then the two upload cells below IN ORDER: porte first, then corridoi (each opens its own file picker).

In [ ]:
from google.colab import files
import zipfile, pathlib

BACKGROUNDS_ROOT = pathlib.Path("non_ostruite")
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}


def upload_venue(venue):
    """Prompts a file picker for one zip and extracts its images (flattened,
    ignoring any nested folder structure inside the zip) into non_ostruite/<venue>."""
    print(f"select the zip for '{venue}'...")
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    dest = BACKGROUNDS_ROOT / venue
    dest.mkdir(parents=True, exist_ok=True)
    n = 0
    with zipfile.ZipFile(zip_name) as zf:
        for member in zf.namelist():
            name = pathlib.Path(member)
            if member.endswith("/") or name.suffix.lower() not in IMAGE_EXTS:
                continue
            (dest / name.name).write_bytes(zf.read(member))
            n += 1
    pathlib.Path(zip_name).unlink()  # don't leave the zip lying around in /content
    print(f"{venue}: extracted {n} images -> {dest}")

### 3a. Upload `porte`

In [ ]:
upload_venue("porte")

### 3b. Upload `corridoi`

In [ ]:
upload_venue("corridoi")

## 4. Config

Edit these to control cost/scope of the run.

In [ ]:
import os

BACKGROUND_VENUES = {
    "porte": str(BACKGROUNDS_ROOT / "porte"),
    "corridoi": str(BACKGROUNDS_ROOT / "corridoi"),
}

SCENE_DESCRIPTION = {
    "porte": "a doorway that is part of a hospital emergency exit route",
    "corridoi": "a hospital corridor that is part of an emergency exit route",
}

# category key -> object description used in the prompt. Keys match the label_class
# values used elsewhere in the DB (see refresh_copypaste_bboxes.py) so the two
# datasets stay comparable.
OBSTACLE_DESCRIPTIONS = {
    "carrello": "a metal hospital utility cart/trolley with shelves",
    "sedia": "an empty wheelchair",
    "barella": "a hospital stretcher (gurney)",
    "scatola": "a stack of two or three cardboard boxes",
    "cestino": "a waste bin / trash can",
}

PROMPT_TEMPLATE = (
    "You are editing a real photograph of {scene}, for a computer-vision "
    "dataset about blocked emergency exits. Add {n_word} on the floor, "
    "blocking the passage shown in the photo: {object_list}. This must be a "
    "MINIMAL, LOCAL edit: only add these objects, do not change anything "
    "else. Keep the exact original image resolution, aspect ratio, framing, "
    "camera angle and zoom level - do not crop, pan, rotate, zoom, letterbox "
    "or re-render the scene. Every pixel outside the added objects (walls, "
    "floor, doors, lighting, shadows already present) must stay identical to "
    "the original photo.\n\n"
    "Placement: scatter the objects naturally, at different distances from "
    "the camera and scaled accordingly (closer = bigger) - do not line them "
    "up neatly or center them. {cut_instruction}\n\n"
    "Each added object must be photorealistic, correctly scaled and "
    "consistent with the scene's existing lighting, perspective and shadows. "
    "Do not add people. Output only the edited photo, no text."
)

CUT_INSTRUCTION = (
    "At least one object should be positioned so it is PARTIALLY CUT OFF by "
    "the edge of the photo (left, right, or bottom border), as if the "
    "photographer didn't perfectly frame the shot - keep it that way, don't "
    "fit everything neatly inside the frame."
)

MODEL = "gemini-2.5-flash-image"  # nicknamed "nano banana"
OUT_DIR = "ostruzioni_gemini"
N_IMAGES = 60          # backgrounds to use (1 edited image per background, capped at
                       # the number of unique backgrounds available across VENUES)
SEED = 42
SLEEP_SECONDS = 1.0    # between API calls, rate limiting
RETRIES = 2            # per image, on API error/refusal
CATEGORIES = list(OBSTACLE_DESCRIPTIONS.keys())   # subset if you want fewer categories
VENUES = list(BACKGROUND_VENUES.keys())           # subset, e.g. ["porte"] only
MIN_OBJECTS = 1        # objects per image is randomized in [MIN_OBJECTS, MAX_OBJECTS]
MAX_OBJECTS = 3
CUT_PROBABILITY = 0.3  # chance an image asks for one object partially cut off by the border
ASPECT_DRIFT_WARN_PCT = 3.0    # flag+log outputs whose ASPECT RATIO drifted more than this
                                # (raw resolution mismatch is expected/harmless: Gemini snaps
                                # to a handful of fixed output sizes, never the source's exact
                                # resolution - only a changed w/h RATIO means real reframing)
DRY_RUN = True         # True = only print sampled jobs, no API calls/cost

## 5. Sampling

Picks at most one job per unique background (never the same background twice), so the output filename can stay identical to the source and unambiguously identify the reference image. If `N_IMAGES` exceeds the number of available backgrounds, it's capped and a warning is printed.

In [ ]:
import glob, random

def load_backgrounds_by_venue():
    backgrounds_by_venue = {}
    for venue, venue_dir in BACKGROUND_VENUES.items():
        paths = []
        for ext in ("*.jpg", "*.jpeg", "*.png"):
            paths.extend(glob.glob(os.path.join(venue_dir, ext)))
        backgrounds_by_venue[venue] = sorted(paths)
        print(f"venue {venue}: {len(paths)} backgrounds found")
    return backgrounds_by_venue


def sample_jobs(backgrounds_by_venue, categories, n, seed,
                 min_objects=1, max_objects=1, cut_probability=0.0):
    """One job per unique background (venue, background). No repeats: a
    background is used at most once so its generated image can keep the exact
    same filename as the source. Each job gets a random number of objects
    (min_objects..max_objects, categories drawn with replacement) and a coin
    flip for whether one of them should be requested partially cut off."""
    all_backgrounds = [
        (venue, bg)
        for venue, bg_paths in backgrounds_by_venue.items()
        for bg in bg_paths
    ]
    if not all_backgrounds or not categories:
        return []

    rng = random.Random(seed)
    shuffled = all_backgrounds[:]
    rng.shuffle(shuffled)
    if n > len(shuffled):
        print(f"[warn] requested n={n} exceeds {len(shuffled)} unique backgrounds "
              f"available - capping to {len(shuffled)} (1 image per background max)")
        n = len(shuffled)

    jobs = []
    for venue, bg in shuffled[:n]:
        count = rng.randint(min_objects, max_objects)
        jobs.append({
            "venue": venue,
            "background": bg,
            "categories": rng.choices(categories, k=count),
            "cut": rng.random() < cut_probability,
        })
    return jobs

## 6. Gemini client + edit call

In [ ]:
import io
from google import genai
from google.genai import types
from PIL import Image

client = genai.Client(vertexai=True, project=VERTEX_PROJECT, location=VERTEX_LOCATION)


_NUM_WORDS = {1: "one object", 2: "two objects", 3: "three objects",
              4: "four objects", 5: "five objects"}


def build_prompt(job):
    descriptions = [OBSTACLE_DESCRIPTIONS[c] for c in job["categories"]]
    n = len(descriptions)
    if n == 1:
        object_list = descriptions[0]
    else:
        object_list = ", ".join(descriptions[:-1]) + " and " + descriptions[-1]
    return PROMPT_TEMPLATE.format(
        scene=SCENE_DESCRIPTION[job["venue"]],
        n_word=_NUM_WORDS.get(n, f"{n} objects"),
        object_list=object_list,
        cut_instruction=CUT_INSTRUCTION if job["cut"] else "",
    )


def call_gemini_edit(bg_image, prompt):
    """Returns (PIL.Image | None, refusal_text | None)."""
    response = client.models.generate_content(
        model=MODEL,
        contents=[prompt, bg_image],
        config=types.GenerateContentConfig(response_modalities=["Image", "Text"]),
    )
    refusal_text = None
    for part in response.candidates[0].content.parts:
        if getattr(part, "inline_data", None) is not None:
            return Image.open(io.BytesIO(part.inline_data.data)), None
        if getattr(part, "text", None):
            refusal_text = part.text
    return None, refusal_text

## 7. Run generation

With `DRY_RUN = True` (default) this only prints the sampled jobs, no API calls, no cost. Flip `DRY_RUN = False` in the config cell once you're happy with the sample, then re-run this cell.

Images are written to `images/<venue>/...` (`porte` vs `corridoi` in separate subfolders), and every output is force-resized back to the EXACT pixel size of its source background - required so P3's patch-level comparison against the memory bank stays aligned. Gemini will basically NEVER return your exact source resolution (it snaps to a handful of fixed output sizes) - that's expected and harmless, since forcing back to the source size only rescales uniformly as long as the ASPECT RATIO is preserved. What actually warns you is a change in aspect ratio (w/h), which means the scene got reframed/cropped rather than just resized, and is worth a manual look.

In [ ]:
import json, time
from datetime import datetime, timezone
from pathlib import Path

backgrounds_by_venue = load_backgrounds_by_venue()
backgrounds_by_venue = {v: p for v, p in backgrounds_by_venue.items() if v in VENUES}

jobs = sample_jobs(backgrounds_by_venue, CATEGORIES, N_IMAGES, SEED,
                    min_objects=MIN_OBJECTS, max_objects=MAX_OBJECTS,
                    cut_probability=CUT_PROBABILITY)
print(f"sampled {len(jobs)} generation jobs (requested n={N_IMAGES})")

out_dir = Path(OUT_DIR)
images_root = out_dir / "images"
for venue in VENUES:
    (images_root / venue).mkdir(parents=True, exist_ok=True)

metadata = {}
failures = []
n_ok = 0
n_resized = 0

for i, job in enumerate(jobs, start=1):
    # identical filename to the source background - this is the key linking the
    # generated test image back to its reference for the P3 memory bank / gating
    out_name = Path(job["background"]).name
    out_key = f"{job['venue']}/{out_name}"
    out_path = images_root / job["venue"] / out_name
    prompt = build_prompt(job)

    if DRY_RUN:
        cut_note = " +cut" if job["cut"] else ""
        print(f"[{i}/{len(jobs)}] DRY-RUN would generate {out_key} <- "
              f"{job['background']} ({', '.join(job['categories'])}{cut_note})")
        continue

    ok, last_error = False, None
    for attempt in range(1, RETRIES + 2):
        try:
            bg_image = Image.open(job["background"]).convert("RGB")
            target_size = bg_image.size  # (w, h) - the exact size we must match on output
            img, refusal = call_gemini_edit(bg_image, prompt)
            if img is not None:
                img = img.convert("RGB")
                out_size = img.size
                resized = out_size != target_size
                if resized:
                    img = img.resize(target_size, Image.Resampling.LANCZOS)
                    n_resized += 1
                # raw resolution mismatch is expected (Gemini's fixed output sizes) and
                # harmless as long as the ASPECT RATIO matches - only ratio drift means
                # the scene was actually reframed/cropped, not just up/downscaled
                src_ar = target_size[0] / target_size[1]
                out_ar = out_size[0] / out_size[1]
                aspect_drift_pct = round(100 * abs(out_ar - src_ar) / src_ar, 2)
                if aspect_drift_pct > ASPECT_DRIFT_WARN_PCT:
                    print(f"[{i}/{len(jobs)}] WARNING {out_key}: aspect ratio drifted "
                          f"{aspect_drift_pct}% (source {target_size} -> raw output {out_size}) "
                          "- likely reframed, review manually")
                # keep the source's own format/extension (jpg stays jpg, png stays png)
                save_kwargs = {"quality": 95} if out_path.suffix.lower() in (".jpg", ".jpeg") else {}
                img.save(out_path, **save_kwargs)
                metadata[out_key] = {
                    "venue": job["venue"],
                    "categories": job["categories"],
                    "cut_requested": job["cut"],
                    "source_background": job["background"],
                    "prompt": prompt,
                    "model": MODEL,
                    "target_size": list(target_size),
                    "raw_output_size": list(out_size),
                    "resized_to_match": resized,
                    "aspect_drift_pct": aspect_drift_pct,
                }
                ok = True
                n_ok += 1
                break
            last_error = refusal or "no image returned"
        except Exception as e:
            last_error = str(e)
        if attempt <= RETRIES:
            time.sleep(SLEEP_SECONDS * attempt)

    if not ok:
        print(f"[{i}/{len(jobs)}] FAILED {out_key}: {last_error}")
        failures.append({**job, "error": last_error})
    elif i % 10 == 0 or i == len(jobs):
        print(f"[{i}/{len(jobs)}] generated {n_ok} so far...")

    time.sleep(SLEEP_SECONDS)

if not DRY_RUN:
    meta_doc = {
        "_meta": {
            "model": MODEL,
            "seed": SEED,
            "n_requested": N_IMAGES,
            "n_generated": n_ok,
            "n_resized_to_match": n_resized,
            "aspect_drift_warn_pct": ASPECT_DRIFT_WARN_PCT,
            "categories": CATEGORIES,
            "min_objects": MIN_OBJECTS,
            "max_objects": MAX_OBJECTS,
            "cut_probability": CUT_PROBABILITY,
            "venues": VENUES,
            "prompt_template": PROMPT_TEMPLATE,
            "generated_at": datetime.now(timezone.utc).isoformat(),
        },
        "images": metadata,
    }
    (out_dir / "objects_metadata.json").write_text(
        json.dumps(meta_doc, indent=1, ensure_ascii=False), encoding="utf-8")
    if failures:
        (out_dir / "failures.json").write_text(
            json.dumps(failures, indent=1, ensure_ascii=False), encoding="utf-8")
    print(f"\ndone: {n_ok}/{len(jobs)} images generated -> {images_root}")
    print(f"{n_resized}/{n_ok} outputs needed a force-resize to match the source size exactly")
    if failures:
        print(f"{len(failures)} failures logged in {out_dir / 'failures.json'}")

## 8. Download the results back to your PC

In [ ]:
import shutil
from google.colab import files

archive_path = shutil.make_archive("ostruzioni_gemini", "zip", out_dir)
files.download(archive_path)

## Next steps (back on your PC)

1. Unzip the downloaded archive into `Dataset/ostruzioni_gemini`. Check `objects_metadata.json` for any image with `aspect_drift_pct` above the warning threshold and eyeball those first - a large drift means Gemini reframed the scene rather than a simple resize.
2. Annotate bounding boxes by hand, one venue at a time (kept separate so you can point P3's ROI/gating tooling only at `porte`):
   ```bash
   python pipeline1/src/annotate_bboxes.py \
       --images "Dataset/ostruzioni_gemini/images/porte/*.jpg" \
       --out-dir Dataset/ostruzioni_gemini/labels/porte
   python pipeline1/src/annotate_bboxes.py \
       --images "Dataset/ostruzioni_gemini/images/corridoi/*.jpg" \
       --out-dir Dataset/ostruzioni_gemini/labels/corridoi
   ```
3. For P3 gating on the `porte` images: gating is looked up by `reference_frame_id` (P3's `evaluate-db` reads the ROI of the REFERENCE/normal frame, not of the occluded one), so as long as each generated frame's `reference_frame_id` points to its original `non_ostruite/porte/<same filename>` background, an existing ROI on that background is picked up automatically - no need to re-annotate the edited image. `Dataset/non_ostruite/porte` currently has ROI only for a handful of frames though (see todo), so annotate any missing ones on the ORIGINAL background with `annotate_door_roi.py` + `import_roi_to_db.py` (once per background, reused for every test image built from it).
4. Load into the DB with `load_dataset_to_db.py` (e.g. `occlusion_type='synthetic_gemini'`), setting `reference_frame_id` from the identical filename (`non_ostruite/<venue>/<name>` -> `ostruzioni_gemini/images/<venue>/<name>`), then run `evaluate_yolo.py` (P1) / `anomaly_detection_patch_core.py evaluate-db` (P3) on it alongside the copy-paste and poli test sets. `venue_type` (porta/corridoio) is inferred from the filename prefix, same convention as the rest of the dataset.